In [ ]:
import glob
import os
from pathlib import Path

# ---------------------------------------------------------------------
# Repository root
# ---------------------------------------------------------------------
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# Define path to folder containing all the CSV sonata files
input_folder = ROOT / "data" / "processed" / "parsed_features"
input_files = sorted(glob.glob(str(input_folder / "*.csv")))

# Define output folders for preprocessed features
feather_output_folder = ROOT / "data" / "processed" / "preprocessed_features"
csv_output_folder = ROOT / "data" / "processed" / "preprocessed_features_csv"

# Ensure output folders exist
os.makedirs(feather_output_folder, exist_ok=True)
os.makedirs(csv_output_folder, exist_ok=True)

In [ ]:
import pandas as pd
import numpy as np
import re

def impute_harmony_for_same_onset(df, harmony_cols):
    """
    Assign harmony features to notes that share the same onset.
    Only fills missing values, preserving existing harmony data.
    """
    for col in harmony_cols:
        missing_indices = df[df[col].isna()].index  # Get indices where harmony feature is missing

        # Assign harmony only where available at the same onset
        df.loc[missing_indices, col] = df.loc[missing_indices].apply(
            lambda row: df[df['Onset'] == row['Onset']][col].dropna().iloc[0]
            if not df[df['Onset'] == row['Onset']][col].dropna().empty else np.nan,
            axis=1
        )
    
    return df


def impute_localkey(df):
    """
    Efficiently fill missing localkey only if both previous and next localkey exist and are the same.
    Uses shift() to check adjacent values.
    """
    missing_indices = df[df['localkey'].isna()].index  # Get indices where localkey is missing

    # Get previous (lag) and next (lead) localkey values
    prev_key = df['localkey'].shift(1)
    next_key = df['localkey'].shift(-1)

    # Fill only where both prev and next exist and are the same
    df.loc[missing_indices, 'localkey'] = df.loc[missing_indices].apply(
        lambda row: prev_key[row.name] if (prev_key[row.name] == next_key[row.name]) and pd.notna(prev_key[row.name]) else np.nan, axis=1
    )

    return df


def majority_vote_localkey(df):
    """
    For remaining missing localkey values, assign the most frequent key
    from the current measure, previous measure, and next measure.
    """
    for index, row in df.iterrows():
        if pd.isna(row['localkey']):  # If localkey is missing
            # Get surrounding measures
            measure_range = df[(df['start_meas'] >= row['start_meas'] - 1) & (df['start_meas'] <= row['start_meas'] + 1)]
            most_common_key = measure_range['localkey'].mode()  # Find the most common key
            
            if not most_common_key.empty:
                df.at[index, 'localkey'] = most_common_key[0]  # Assign the most frequent localkey

    return df


# Define harmony-related columns
harmony_features = ['shorthand', 'extended', 'majmin', 'quality', 'inversion', 'roman']

# Function to track missing indices before and after imputation and log statistics
def track_imputation(df, impute_func, cols=None, func_name="Imputation"):
    """
    Track missing indices before imputation, apply the imputation function, and log statistics.
    """
    if cols:
        missing_before = {col: df[df[col].isna()].index.tolist() for col in cols}
    else:
        missing_before = df[df.isna()].index.tolist()

    df_imputed = impute_func(df)

    if cols:
        missing_after = {col: df_imputed[df_imputed[col].isna()].index.tolist() for col in cols}
    else:
        missing_after = df_imputed[df_imputed.isna()].index.tolist()

    imputed_stats = {col: len(missing_before[col]) - len(missing_after[col]) for col in cols} if cols else None

    print(f"\n===== {func_name} =====")
    print("Missing Before:\n", missing_before)
    print("Missing After:\n", missing_after)
    print("Imputation Stats:\n", imputed_stats)

    return df_imputed

# Function to extract time signature components
def extract_time_signature(ts):
    try:
        numerator, denominator = map(int, ts.split('/'))
        return pd.Series([numerator, denominator, numerator / denominator])
    except:
        return pd.Series([np.nan, np.nan, np.nan])  # Handle missing values
    
# Mapping note names to their ordinal position in the Circle of Fifths
circle_of_fifths_major = {
    'C': 0, 'G': 1, 'D': 2, 'A': 3, 'E': 4, 'B': 5, 'F#': 6, 'C#': 7, 'G#': 8, 'D#': 9, 'A#': 10, 'F': -1, 
    'Bb': -2, 'Eb': -3, 'Ab': -4, 'Db': -5, 'Gb': -6, 'Cb': -7
}

circle_of_fifths_minor = {
    'A': 0, 'E': 1, 'B': 2, 'F#': 3, 'C#': 4, 'G#': 5, 'D#': 6, 'A#': 7, 'F': -1, 'C': -2, 'G': -3, 'D': -4, 
    'Bb': -5, 'Eb': -6, 'Ab': -7, 'Db': -8, 'Gb': -9
}

# Function to extract numerical key information using the Circle of Fifths
def encode_localkey(localkey):
    if pd.isna(localkey):
        return pd.Series([np.nan, np.nan])  # Handle missing values
    match = re.match(r"([A-Ga-g#b]+):([a-z]+)", localkey)
    if match:
        root, mode = match.groups()
        root = root.replace("♯", "#").replace("♭", "b")  # Normalize accidental notation
        
        if mode == "maj":
            degree = circle_of_fifths_major.get(root, np.nan)
            mode_flag = 1  # Major = 1
        else:
            degree = circle_of_fifths_minor.get(root, np.nan)
            mode_flag = 0  # Minor = 0
            
        return pd.Series([degree, mode_flag])
    
    return pd.Series([np.nan, np.nan])

non_accent_articulations = {'staccato', 'tenuto'}

# Define ordinal mapping for dynamics
ordinal_dynamics = {'ppp': 1, 'pp': 2, 'p': 3, 'mp': 4, 'mf': 5, 'f': 6, 'ff': 7}

# Define articulations and dynamics that indicate an accent
accent_triggers = {'accent', 'strong accent'}
dynamic_accent_triggers = {'sf', 'rfz', 'sfp', 'rf'}

# Define semitone mappings for major and minor scales
scale_degree_to_semitones = {
    "major": {1: 0, 2: 2, 3: 4, 4: 5, 5: 7, 6: 9, 7: 11, 
              "bII": 1, "bIII": 3, "bVI": 8, "bVII": 10},  
    "minor": {1: 0, 2: 2, 3: 3, 4: 5, 5: 7, 6: 8, 7: 10, 
              "bII": 1, "bIII": 3, "bVI": 8, "bVII": 10}  
}

# Map inversion labels to numeric index (0 = root, 1 = first, etc.)
inversion_index_mapping = {
    '': 0,     # Root position
    '6': 1,    # First inversion (triad)
    '64': 2,   # Second inversion (triad)
    '65': 1,   # First inversion (7th)
    '43': 2,   # Second inversion (7th)
    '42': 3    # Third inversion (7th)
}

# Scale degree mappings for standard Roman numerals, including flats
scale_degree_mapping = {
    'I': 1, 'i': 1, 'ii': 2, 'ii-': 2, 'ii=': 2, 'iii': 3, 'III': 3, 'IV': 4, 'iv': 4, 
    'V': 5, 'v': 5, 'VI': 6, 'vi': 6, 'VII': 7, 'vii': 7, 
    'bII': "bII", 'bIII': "bIII", 'bVI': "bVI", 'bVII': "bVII"
}

# Chord Quality Mapping (One-Hot)
chord_quality_mapping = {
    "M": (1, 0, 0, 0), "m": (0, 1, 0, 0), "d": (0, 0, 1, 0), "a": (0, 0, 0, 1),
    "M7": (1, 0, 0, 0), "m7": (0, 1, 0, 0), "d7": (0, 0, 1, 0), "h7": (0, 0, 1, 0),
    "D7": (1, 0, 0, 0), "a6": (0, 0, 0, 1)
}

def encode_roman_numeral(roman, quality, localkey, majmin):
    if pd.isna(roman) or pd.isna(localkey):
        return pd.Series([-1] * 10)

    mode = "major" if "maj" in majmin else "minor"

    # Extract root Roman numeral
    match = re.match(r"([IVivb]+)", roman)
    root = match.group(1) if match else None

    # Extract inversion label and index
    inversion_match = re.search(r"(65|43|42|6|7|64)", roman)
    inversion_label = inversion_match.group(1) if inversion_match else ''
    inversion_index = inversion_index_mapping.get(inversion_label, -1)

    # Extract secondary dominant
    sec_dom_match = re.search(r"/([IViv]+)", roman)
    secondary_root = sec_dom_match.group(1) if sec_dom_match else None

    # Special chords
    is_aug_sixth = int(any(x in roman for x in ['It+6', 'Fr+6', 'Gr+6']))
    is_neapolitan = int('N6' in roman)
    is_seventh = int(quality in ['D7', 'm7', 'M7', 'h7', 'd7'])

    # Override inversion for special chords
    if is_aug_sixth or is_neapolitan:
        inversion_index = -1

    # Scale degree → semitone offset
    scale_degree = scale_degree_mapping.get(root, -1)
    semitone_offset = scale_degree_to_semitones[mode].get(scale_degree, -1)

    # Secondary scale degree semitone
    if secondary_root:
        sec_scale_degree = scale_degree_mapping.get(secondary_root, -1)
        secondary_semitone_offset = scale_degree_to_semitones[mode].get(sec_scale_degree, -1)
    else:
        secondary_semitone_offset = -1

    # One-hot chord quality
    is_major, is_minor, is_diminished, is_augmented = chord_quality_mapping.get(quality, (0, 0, 0, 0))

    if sum([is_major, is_minor, is_diminished, is_augmented]) != 1:
        raise ValueError(f"Chord quality not properly assigned: {roman}, {quality}")

    return pd.Series([
        semitone_offset,
        secondary_semitone_offset,
        is_major, is_minor, is_diminished, is_augmented,
        is_seventh,
        inversion_index,
        is_aug_sixth,
        is_neapolitan
    ])

# Column headers for encoded output
encoded_columns = [
    'scale_degree_semitone_offset', 'secondary_scale_degree_semitone_offset',
    'is_major', 'is_minor', 'is_diminished', 'is_augmented',
    'is_seventh', 'chord_inversion',
    'is_augmented_sixth', 'is_neapolitan'
]

# Define the mapping for harmonic complexity based on chord quality
harmonic_complexity_mapping = {
    'M': 1, 'm': 1,  # Major and minor triads (simple chords)
    'd': 2, 'a': 2,  # Diminished and augmented (increased tension)
    'D7': 3, 'd7': 3, 'M7': 3, 'm7': 3, 'h7': 3,  # Seventh chords (additional harmonic layer)
    'a6': 4  # Augmented sixth chords (highly chromatic and complex)
}

# Define tension alterations and their impact (only if not already implied by the chord quality)
tension_mapping = {
    'b7': 1,   # Dominant seventh adds moderate tension
    'bb7': 2,  # Double flatted seventh increases tension significantly
    'b5': 2    # Flattened fifth increases harmonic instability (but not for diminished)
}

# Function to compute harmonic complexity based on both quality and alterations in 'extended'
def compute_harmonic_complexity(row):
    quality = row['quality']
    extended = str(row['extended'])

    # Start with the complexity level from the quality mapping
    complexity = harmonic_complexity_mapping.get(quality, 1)

    implied_alterations = set()
    if quality in ['D7', 'M7', 'm7', 'h7', 'd7']:
        implied_alterations.add('b7')
    if quality == 'd':
        implied_alterations.add('b5')

    for alteration, impact in tension_mapping.items():
        if alteration in extended and alteration not in implied_alterations:
            complexity += impact
    # Adjust complexity based on the presence of tension-altering notes **only if they are not inherent**
    for alteration, impact in tension_mapping.items():
        if alteration in extended:
            # **Skip b5 if the chord is already diminished (d)**
            if alteration == 'b5' and quality == 'd':
                continue  # Diminished chords always contain b5, so no need to add extra tension
            complexity += impact  # Increase complexity based on alterations

    return complexity

# Direct Mapping of Local Key to MIDI Pitch (Tonic)
tonic_midi_mapping = {
    'C:maj': 60, 'G:maj': 67, 'D:maj': 62, 'A:maj': 69, 'E:maj': 64, 'B:maj': 71, 'F#:maj': 66, 'C#:maj': 61,
    'G#:maj': 68, 'D#:maj': 63, 'A#:maj': 70, 'F:maj': 65, 'Bb:maj': 70, 'Eb:maj': 63, 'Ab:maj': 68, 
    'Db:maj': 61, 'Gb:maj': 66, 'Cb:maj': 59,

    'A:min': 57, 'E:min': 64, 'B:min': 59, 'F#:min': 66, 'C#:min': 61, 'G#:min': 68, 'D#:min': 63, 'A#:min': 70,
    'F:min': 65, 'C:min': 60, 'G:min': 67, 'D:min': 62, 'Bb:min': 58, 'Eb:min': 63, 'Ab:min': 68, 'Db:min': 61, 'Gb:min': 66
}

# Function to convert fraction-like strings to floats
def fraction_to_float(value):
    try:
        return eval(value) if '/' in value else float(value)
    except:
        return float('nan')  # Handle invalid cases
    
def save_df_as_feather(df, folder_name, file_name):
    # Create the folder if it does not exist
    folder_name.mkdir(parents=True, exist_ok=True)
    
    # Define the full file path
    file_path = folder_name / f"{file_name}.feather"
    
    # Save DataFrame as a Feather file
    df.to_feather(file_path)
    
    print(f"DataFrame saved successfully at {file_path}")


def save_df_as_csv(df, folder_name, file_name):
    # Create the folder if it does not exist
    folder_name.mkdir(parents=True, exist_ok=True)
    
    # Define the full file path
    file_path = folder_name / f"{file_name}.csv"
    
    # Save DataFrame as a CSV file
    df.to_csv(file_path, index=False)
    
    print(f"DataFrame saved successfully at {file_path}")

In [ ]:
for file_path in input_files:
    file_name = Path(file_path).stem
    print(f"\n=== Processing {file_name} ===")

    df = pd.read_csv(file_path)

    # === Fix duration and onset columns bug ===
    df['duration_quarterLength'] = pd.to_numeric(df['duration_quarterLength'], errors='coerce')
    df['Onset'] = pd.to_numeric(df['Onset'], errors='coerce')
    df['pitch'] = pd.to_numeric(df['pitch'], errors='coerce')
    df['Beat'] = pd.to_numeric(df['Beat'], errors='coerce')


    df_notes = df[df['Type'] != "Rest"].copy()

    # === Optional: silent imputation ===
    df_notes = impute_harmony_for_same_onset(df_notes, harmony_features)
    df_notes = impute_localkey(df_notes)
    df_notes = majority_vote_localkey(df_notes)

    df.loc[df['Type'] != "Rest", df_notes.columns] = df_notes.values

    df[['timeSig_numerator', 'timeSig_denominator', 'timeSig_ratio']] = df['timeSig'].apply(extract_time_signature)
    df_notes[['timeSig_numerator', 'timeSig_denominator', 'timeSig_ratio']] = df_notes['timeSig'].apply(extract_time_signature)

    df[['localkey_fifths', 'localkey_mode']] = None
    df_notes[['localkey_fifths', 'localkey_mode']] = df_notes['localkey'].apply(encode_localkey)
    df.loc[df['Type'] != "Rest", df_notes.columns] = df_notes.values

    for articulation in non_accent_articulations:
        df[f'is_{articulation}'] = df['articulation'].apply(
            lambda x: 1 if pd.notna(x) and articulation in x.split(', ') else 0
        )

    df['is_Ornament'] = df['Ornaments'].notna().astype(int)

    df['is_accent'] = df.apply(
        lambda row: int(any(art in row['articulation'].split(', ') for art in accent_triggers)
                       if pd.notna(row['articulation']) else 0) 
                    or int(row['Dynamic'] in dynamic_accent_triggers),
        axis=1
    )

    df['ordinal_dynamic'] = df['Dynamic'].map(ordinal_dynamics).fillna(0).astype(int)
    df.loc[df['Dynamic'] == 'sfp', 'ordinal_dynamic'] = ordinal_dynamics['p']
    df.loc[df['Dynamic'] == 'rf', 'ordinal_dynamic'] = ordinal_dynamics['f']

    df['is_cresc'] = (df['Cresc_ID'] != 0).astype(int)
    df['is_dim'] = (df['Dim_ID'] != 0).astype(int)
    df['is_slur'] = (df['Slur_ID'] != 0).astype(int)

    df[encoded_columns] = df.apply(
        lambda row: encode_roman_numeral(row['roman'], row['quality'], row['localkey'], row['majmin']),
        axis=1
    )

    df['harmonic_complexity'] = df.apply(compute_harmonic_complexity, axis=1)

    df['tonic_midi'] = df['localkey'].map(tonic_midi_mapping)
    df['relative_pitch_scale'] = df['pitch'] - df['tonic_midi']
    df['octave'] = df['pitch'] // 12

    df['IOI'] = df['Onset'].diff().fillna(0)
    df["Beat"] = df["Beat"].astype(str).apply(fraction_to_float).astype("float64")
    df['scaled_duration'] = df['duration_quarterLength'] / df['timeSig_ratio']
    df['scaled_beat'] = df['Beat'] / df['timeSig_ratio']
    df['normalized_IOI'] = df['IOI'] / df['timeSig_ratio']

    df.columns = df.columns.str.lower()
    df["chord_inversion"] = pd.to_numeric(df["chord_inversion"], errors="coerce").astype("Int64")

    print("Feature Normalization & Standardization Completed.")
    # print(df.info())

    # Save as feather
    save_df_as_feather(df, feather_output_folder, file_name)
    # Save as CSV
    save_df_as_csv(df, csv_output_folder, file_name)